In [1]:
import os

PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/Rag project'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir(PROJECT_DIR)
except ImportError:
    print('Not running in Colab: using the current directory')

print('Working directory:', os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/MyDrive/Colab Notebooks/Rag project


In [11]:
import json
import os
import re
import torch
import yaml
import pandas as pd
from pathlib import Path
import utils

CORPUS_FILE = 'corpus_2026-10-09.jsonl'
QUESTIONS_FILE = 'vlm_eval_questions_combined.xlsx'
RUN_FILE = 'retrieval_eval/run_bm25+qwen3.json'   # or a reranked run
OUT_DIR = 'generation_eval'

MODEL_NAME = 'Qwen/Qwen3-8B'      # 'Qwen/Qwen3-4B' if GPU memory is tight
LOAD_IN_4BIT = None               # None = auto (4-bit if the GPU has < 20 GB)
TOP_K = 5                         # abstracts given to the model
MAX_NEW_TOKENS = 400
DRY_RUN = False                   # True = skip the model, test the pipeline with dummy answers

ABSTAIN_TOKEN = 'NO_RELEVANT_STUDIES'
os.makedirs(OUT_DIR, exist_ok=True)



In [7]:
_cfg = yaml.safe_load(Path("config.yml").read_text())
ST_CONFIGS = _cfg["ST_CONFIGS"]
MEDCPT = _cfg["MEDCPT"]

# Inject configurations and device into the imported utils module
utils.ST_CONFIGS = ST_CONFIGS
utils.MEDCPT = MEDCPT


In [12]:
 corpus = utils.load_corpus(CORPUS_FILE)

qdf = pd.read_excel(QUESTIONS_FILE, dtype=str).fillna('')
qdf = qdf[qdf.row_type == 'question'].reset_index(drop=True)
split = lambda s: [p for p in s.split(';') if p]
qdf['seeds'] = qdf.seed_pmids.map(split)
qdf['traps'] = qdf.fp_target_pmids.map(split)

with open(RUN_FILE) as f:
    run = json.load(f)  # ranx JSON: {qid: {pmid: score}}

def top_k(qid, k=TOP_K):
    return [p for p, _ in sorted(run[qid].items(), key=lambda x: -x[1])[:k]]

print(f'{len(corpus)} docs | {len(qdf)} questions | run: {RUN_FILE}')



100 docs | 48 questions | run: retrieval_eval/run_bm25+qwen3.json


In [13]:
SYSTEM_PROMPT = f"""You are a careful biomedical research assistant.
Answer the question using ONLY the abstracts provided. Do not use outside knowledge.

Rules:
- Cite every claim with the PMID in square brackets, e.g. [42853796].
- Cite only PMIDs from the provided abstracts.
- If an abstract is only loosely related, do not use it.
- If none of the abstracts answer the question, reply with exactly:
  {ABSTAIN_TOKEN}
  followed by one sentence explaining why.
- Be concise: at most 150 words."""


def build_messages(question, pmids):
    context = '\n\n'.join(
        f"[{p}] {corpus[p].get('title') or ''}\n{corpus[p]['abstract']}" for p in pmids)
    user = f'Abstracts:\n\n{context}\n\nQuestion: {question}'
    return [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user}]


In [14]:

if not DRY_RUN:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if LOAD_IN_4BIT is None:
        gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
        LOAD_IN_4BIT = gpu_gb < 20
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_compute_dtype=torch.float16) if LOAD_IN_4BIT else None

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map='auto', quantization_config=quant,
        torch_dtype=None if quant else 'auto').eval()
    print(f'Loaded {MODEL_NAME} (4-bit: {LOAD_IN_4BIT})')


def generate(messages):
    if DRY_RUN:  # dummy answer citing the first retrieved PMID
        first = re.search(r'\[(\d+)\]', messages[1]['content']).group(1)
        return f'Dummy answer based on one study [{first}].'
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)  # direct answers, no reasoning trace
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)  # greedy = reproducible
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-8B (4-bit: False)


In [15]:
results = []
for i, q in enumerate(qdf.itertuples(), 1):
    context = top_k(q.qid)
    answer = generate(build_messages(q.question, context))
    results.append({'qid': q.qid, 'type': q.type, 'question': q.question,
                    'context_pmids': context, 'answer': answer})
    print(f'[{i}/{len(qdf)}] {q.qid}: {answer[:100]!r}')

with open(os.path.join(OUT_DIR, 'answers.jsonl'), 'w') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')



[1/48] q01: 'In the reader study scoring presumptive septic carpal arthritis on calf radiographs, ChatGPT-5 agree'
[2/48] q02: 'ChatGPT-5 achieved an overall diagnostic accuracy of 52% (131/250) and Gemini 2.5 Pro achieved 46% ('
[3/48] q03: 'Gemini was most accurate at classifying dentition stage on panoramic radiographs, achieving 92.8% ac'
[4/48] q04: 'In the 20-center prospective study, AI-Gemini 2.5 achieved 94.2% three-class diagnostic accuracy for'
[5/48] q05: 'Claude Opus 4.7 was more accurate for the four-class diagnosis of oral autoimmune blistering disease'
[6/48] q06: 'The RadGraph F1 for the Findings sections generated by RadVLM was 0.251 [42723869].'
[7/48] q07: 'The model that achieved the highest overall accuracy on retinal multiple-choice questions derived fr'
[8/48] q08: 'ChatGPT-4o achieved the highest diagnostic accuracy of 75% (603/800) in diagnosing retinal diseases '
[9/48] q09: 'The multimodal LLM that performed best for benign-versus-malignant classification of

In [16]:
rows = []
for r, q in zip(results, qdf.itertuples()):
    cited = list(dict.fromkeys(re.findall(r'\[(\d{6,9})\]', r['answer'])))
    abstained = ABSTAIN_TOKEN in r['answer']
    should_abstain = q.type == 'unanswerable'
    rows.append({
        'qid': q.qid, 'type': q.type,
        'abstained': abstained,
        'abstention_correct': abstained == should_abstain,
        'n_cited': len(cited),
        'cited_outside_context': [p for p in cited if p not in r['context_pmids']],  # invented citations
        'cited_traps': [p for p in cited if p in q.traps],
        'seeds_in_context': sum(p in r['context_pmids'] for p in q.seeds),
        'seeds_cited': sum(p in cited for p in q.seeds),
        'n_seeds': len(q.seeds),
    })
checks = pd.DataFrame(rows)
checks.to_csv(os.path.join(OUT_DIR, 'auto_checks.csv'), index=False)

answerable = checks[checks.type != 'unanswerable']
unans = checks[checks.type == 'unanswerable']
trap_q = checks[qdf.traps.map(len).values > 0]
summary = {
    'abstention: correct on unanswerable': f'{unans.abstained.sum()}/{len(unans)}',
    'abstention: wrongly abstained on answerable': f'{answerable.abstained.sum()}/{len(answerable)}',
    'answers with a citation outside the context': f'{(checks.cited_outside_context.map(len) > 0).sum()}/{len(checks)}',
    'trap questions where a trap was cited': f'{(trap_q.cited_traps.map(len) > 0).sum()}/{len(trap_q)}',
    'seed citation rate (seeds cited / seeds retrieved)':
        f'{answerable.seeds_cited.sum()}/{answerable.seeds_in_context.sum()}',
}
print('\n'.join(f'{k}: {v}' for k, v in summary.items()))
with open(os.path.join(OUT_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

abstention: correct on unanswerable: 4/4
abstention: wrongly abstained on answerable: 1/44
answers with a citation outside the context: 0/48
trap questions where a trap was cited: 1/8
seed citation rate (seeds cited / seeds retrieved): 76/78
